In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import WeightedRandomSampler
import timm
import os
from PIL import Image

# Dataset class (modified to handle test data without labels)
class RegionIDImageDataset(Dataset):
    def __init__(self, image_dir, csv_file=None, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        if csv_file:
            self.data = pd.read_csv(csv_file)
            self.region_id_to_idx = {rid: idx for idx, rid in enumerate(sorted(self.data['Region_ID'].unique()))}
            self.data['Region_Index'] = self.data['Region_ID'].map(self.region_id_to_idx)
        else:
            self.data = pd.DataFrame({'filename': sorted(os.listdir(image_dir))})
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['filename']
        img_path = os.path.join(self.image_dir, filename)
        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if 'Region_Index' in row:
            return image, torch.tensor(int(row['Region_Index']), dtype=torch.long)
        return image, filename

# Model with ResNet-50 (unchanged)
class RegionClassificationModel(nn.Module):
    def __init__(self, num_classes):
        super(RegionClassificationModel, self).__init__()
        self.backbone = timm.create_model('convnext_base', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Transforms (unchanged)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Data paths
CSV_FILE = "/kaggle/input/phase2data/Phase_2_data-20250414T180449Z-001/Phase_2_data/labels_train.csv"        
IMAGE_DIR = "/kaggle/input/phase2data/Phase_2_data-20250414T180449Z-001/Phase_2_data/images_train/images_train" 
VAL_CSV_FILE = "/kaggle/input/phase2data/Phase_2_data-20250414T180449Z-001/Phase_2_data/labels_val.csv"  
VAL_IMAGE_DIR = "/kaggle/input/phase2data/Phase_2_data-20250414T180449Z-001/Phase_2_data/images_val"  
TEST_DIR = "/kaggle/input/smai-test/images_test"

# Datasets and Dataloaders
train_dataset = RegionIDImageDataset(csv_file=CSV_FILE, image_dir=IMAGE_DIR, transform=train_transform)
val_dataset = RegionIDImageDataset(csv_file=VAL_CSV_FILE, image_dir=VAL_IMAGE_DIR, transform=val_transform)
test_dataset = RegionIDImageDataset(image_dir=TEST_DIR, transform=val_transform)

# Weighted sampler for class imbalance
class_counts = train_dataset.data['Region_ID'].value_counts()
weights = [1.0 / class_counts[rid] for rid in train_dataset.data['Region_ID']]
sampler = WeightedRandomSampler(weights, num_samples=len(train_dataset), replacement=True)
train_dataloader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Model and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RegionClassificationModel(num_classes=len(train_dataset.region_id_to_idx))
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model = model.to(device)

# No freezing - all layers are trainable
# Weighted loss
classes = sorted(train_dataset.data['Region_ID'].unique())
class_weights = compute_class_weight('balanced', classes=classes, y=train_dataset.data['Region_ID'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3, threshold=0.001)

# Training Loop with Progress Percentage
def calculate_accuracy(preds, labels):
    predicted_classes = torch.argmax(preds, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return correct / total

best_val_acc = 0
patience = 8
counter = 0
for epoch in range(30):
    # Training Phase
    model.train()
    total_loss = 0
    total_train_accuracy = 0
    total_batches = len(train_dataloader)
    for i, (imgs, regionid) in enumerate(train_dataloader):
        imgs, regionid = imgs.to(device), regionid.to(device)
        preds = model(imgs)
        loss = loss_fn(preds, regionid)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_train_accuracy += calculate_accuracy(preds, regionid) * imgs.size(0)
        
        # Print progress
        progress = (i + 1) / total_batches * 100
        print(f"\rEpoch {epoch+1} Training: {progress:.1f}%", end="")
    print()  # Newline after training progress
    avg_train_loss = total_loss / len(train_dataloader.dataset)
    avg_train_acc = total_train_accuracy / len(train_dataloader.dataset)

    # Validation Phase
    model.eval()
    total_val_loss = 0
    total_val_accuracy = 0
    total_val_batches = len(val_dataloader)
    with torch.no_grad():
        for i, (imgs, regionid) in enumerate(val_dataloader):
            imgs, regionid = imgs.to(device), regionid.to(device)
            preds = model(imgs)
            val_loss = loss_fn(preds, regionid)
            total_val_loss += val_loss.item() * imgs.size(0)
            total_val_accuracy += calculate_accuracy(preds, regionid) * imgs.size(0)
            
            # Print progress
            progress = (i + 1) / total_val_batches * 100
            print(f"\rEpoch {epoch+1} Validation: {progress:.1f}%", end="")
    print()  # Newline after validation progress
    avg_val_loss = total_val_loss / len(val_dataloader.dataset)
    avg_val_acc = total_val_accuracy / len(val_dataloader.dataset)

    print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Accuracy: {avg_train_acc*100:.2f}%   Validation Loss: {avg_val_loss:.4f}, Accuracy: {avg_val_acc*100:.2f}%")
    
    scheduler.step(avg_val_acc)
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        counter = 0
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping triggered")
        break

# Generate predictions for validation and test sets
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()
idx_to_region_id = {idx: rid for rid, idx in val_dataset.region_id_to_idx.items()}
val_predictions = []
test_predictions = []

# Validation predictions
total_val_accuracy = 0
with torch.no_grad():
    for imgs, regionid in val_dataloader:
        imgs, regionid = imgs.to(device), regionid.to(device)
        preds = model(imgs)
        total_val_accuracy += calculate_accuracy(preds, regionid) * imgs.size(0)
        predicted_classes = torch.argmax(preds, dim=1).cpu().numpy()
        val_predictions.extend([idx_to_region_id[idx] for idx in predicted_classes])

# Test predictions
test_filenames = []
with torch.no_grad():
    for imgs, filenames in test_dataloader:
        imgs = imgs.to(device)
        preds = model(imgs)
        predicted_classes = torch.argmax(preds, dim=1).cpu().numpy()
        test_predictions.extend([idx_to_region_id[idx] for idx in predicted_classes])
        test_filenames.extend(filenames)

final_val_acc = total_val_accuracy / len(val_dataloader.dataset)
print(f"Loaded model validation accuracy: {final_val_acc*100:.2f}%")

# Create output DataFrame
val_df = pd.DataFrame({
    'id': range(len(val_predictions)),
    'Region_ID': val_predictions,
})
test_df = pd.DataFrame({
    'id': range(len(val_predictions), len(val_predictions) + len(test_predictions)),
    'Region_ID': test_predictions,
})
output_df = pd.concat([val_df, test_df], ignore_index=True)
output_df.to_csv('/kaggle/working/predictions.csv', index=False)
print("Predictions saved to /kaggle/working/predictions.csv")
print(os.listdir('/kaggle/working/'))